<a href="https://colab.research.google.com/github/alirezakiann/snappfood-sentiment/blob/main/notebooks/06_dashboard_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 - Data for the Power BI Dashboard
Collect everything from notebooks 02-05 into a few tidy CSV files that Power BI can read directly.

| File | One row per | Used for |
|---|---|---|
| `reviews.csv` | review | overview, complaint types, model check |
| `food_mentions.csv` | review x food mentioned | complaint rate by food |
| `model_results.csv` | model | model comparison |
| `audit.csv` | hand-labeled review | label-quality chart |

## 0. Setup

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
BASE_DIR = "/content/drive/MyDrive/snappfood-sentiment"
PROC_DIR = f"{BASE_DIR}/processed"
MODEL_DIR = f"{BASE_DIR}/models"
OUT_DIR = f"{BASE_DIR}/powerbi"
os.makedirs(OUT_DIR, exist_ok=True)

for p in [f"{PROC_DIR}/train.csv", f"{BASE_DIR}/complaints_clustered.csv",
          f"{BASE_DIR}/audit_labels.csv", f"{MODEL_DIR}/baseline_tfidf_model.joblib"]:
    print(os.path.basename(p), "found:", os.path.exists(p))

Mounted at /content/drive
train.csv found: True
complaints_clustered.csv found: True
audit_labels.csv found: True
baseline_tfidf_model.joblib found: True


In [2]:
import numpy as np
import pandas as pd
import joblib

## 1. Reviews table

In [3]:
parts = []
for split in ["train", "val", "test"]:
    d = pd.read_csv(f"{PROC_DIR}/{split}.csv")
    d["split"] = split
    parts.append(d)
reviews = pd.concat(parts, ignore_index=True)
reviews["clean_text"] = reviews["clean_text"].fillna("")
reviews.insert(0, "review_id", range(1, len(reviews) + 1))
reviews["sentiment_name"] = reviews["sentiment"].map({1: "Positive", 0: "Negative"})

# review length buckets (with an order column for sorting in Power BI)
reviews["n_words"] = reviews["comment"].astype(str).str.split().str.len()
bins = [0, 5, 10, 20, 40, 10_000]
names = ["1-5", "6-10", "11-20", "21-40", "41+"]
reviews["length_bucket"] = pd.cut(reviews["n_words"], bins=bins, labels=names).astype(str)
reviews["length_bucket_order"] = reviews["length_bucket"].map({n: i for i, n in enumerate(names)})

# simple flags
reviews["mentions_late"] = reviews["comment"].str.contains(r"\bدیر\b|تاخیر|تأخیر", regex=True).astype(int)
reviews["mentions_cold"] = reviews["comment"].str.contains(r"\bسرد\b|یخ کرده", regex=True).astype(int)
latin = reviews["comment"].astype(str).str.count(r"[A-Za-z]")
reviews["is_english"] = (latin / reviews["comment"].astype(str).str.len().clip(lower=1) > 0.5).astype(int)

print(reviews.shape)
reviews[["review_id", "comment", "sentiment_name", "split", "length_bucket", "mentions_late", "is_english"]].head()

(69100, 13)


,review_id,comment,sentiment_name,split,length_bucket,mentions_late,is_english
0,1,راضیم ازتون مثل همیشه …سربلند باشید,Positive,train,6-10,0,0
1,2,صابونی که فرستاده بودند پنپش شکسته و غیر قابل ...,Negative,train,11-20,0,0
2,3,پرتغال پوسیده بود. انار سفید و کاملا ترش. گلاب...,Negative,train,21-40,0,0
3,4,مثل همیشه تازه و خوشمزه ممنون,Positive,train,6-10,0,0
4,5,سفارش ما بعد از ۲ ساعت رسید دستمان,Negative,train,6-10,0,0


## 2. Add complaint types from notebook 05

In [4]:
clusters = pd.read_csv(f"{BASE_DIR}/complaints_clustered.csv")

# in case the names were not saved in notebook 05, apply them here (edit if you chose other names)
CLUSTER_NAMES = {
    0: "Bad taste", 1: "Low quality", 2: "Wrong item sent", 3: "Very bad / small portion",
    4: "Fries & sides", 5: "Bakery, groceries & packaging", 6: "Missing items", 7: "Late & cold delivery",
}
if clusters["complaint_type"].astype(str).str.startswith("cluster").all() or clusters["complaint_type"].isna().all():
    clusters["complaint_type"] = clusters["cluster"].map(CLUSTER_NAMES)

reviews = reviews.merge(clusters[["comment", "cluster", "complaint_type"]].drop_duplicates("comment"),
                        on="comment", how="left")
print("negative reviews with a complaint type:", reviews["complaint_type"].notna().sum())
reviews.loc[reviews["sentiment"] == 0, "complaint_type"].value_counts(dropna=False)

negative reviews with a complaint type: 34506


,count
complaint_type,
Bad taste,9600
"Bakery, groceries & packaging",7825
Late & cold delivery,4321
Low quality,3912
Very bad / small portion,3542
Fries & sides,1814
Wrong item sent,1786
Missing items,1706
NaN,258


## 3. Add the baseline model's predictions on the test set

In [5]:
baseline = joblib.load(f"{MODEL_DIR}/baseline_tfidf_model.joblib")
is_test = reviews["split"] == "test"

reviews["predicted_name"] = ""
reviews.loc[is_test, "predicted_name"] = pd.Series(baseline.predict(reviews.loc[is_test, "clean_text"]),
                                                   index=reviews.index[is_test]).map({1: "Positive", 0: "Negative"})
reviews["model_correct"] = np.where(is_test, (reviews["predicted_name"] == reviews["sentiment_name"]).astype(int), np.nan)

print(f"test accuracy (should be 0.8641): {reviews.loc[is_test, 'model_correct'].mean():.4f}")

test accuracy (should be 0.8641): 0.8641


## 4. Food mentions (long table)
One row for each food a review mentions. A review about pizza **and** fries appears twice.
Fix from notebook 05: «مرغ» no longer matches «تخم مرغ» (eggs); eggs get their own group.

In [6]:
FOODS = {
    "Pizza": r"پیتزا", "Burger": r"برگر", "Sandwich": r"ساندویچ", "Kebab": r"کباب", "Joojeh": r"جوجه",
    "Chicken / fried": r"(?<!تخم )(?<!تخم\u200c)(?<!تخم)مرغ|سوخاری", "Eggs": r"تخم ?\u200c?مرغ",
    "Pasta": r"پاستا|ماکارونی", "Salad": r"سالاد", "Sushi": r"سوشی", "Cake / pastry": r"کیک|شیرینی",
    "Coffee / drinks": r"قهوه|نوشیدنی|نوشابه", "Fries": r"سیب ?\u200c?زمینی", "Rice": r"برنج|چلو",
}
rows = []
for food, pat in FOODS.items():
    hit = reviews[reviews["comment"].str.contains(pat, regex=True)]
    rows.append(hit[["review_id", "sentiment_name", "sentiment"]].assign(food=food))
food_mentions = pd.concat(rows, ignore_index=True)

check = food_mentions.groupby("food").agg(reviews=("review_id", "count"),
                                          complaint_rate=("sentiment", lambda s: (s == 0).mean()))
check.sort_values("complaint_rate", ascending=False).round(3)

,reviews,complaint_rate
food,,
Eggs,350,0.709
Chicken / fried,5162,0.655
Coffee / drinks,1353,0.643
Rice,2046,0.630
Cake / pastry,4381,0.610
Pizza,7648,0.606
Kebab,2260,0.604
Fries,3140,0.603
Pasta,504,0.599


## 5. Model results and audit tables

In [7]:
model_results = pd.DataFrame([
    {"model": "Always guess one class",        "accuracy": 0.500,  "macro_f1": None,   "order": 1},
    {"model": "BiLSTM (PyTorch)",              "accuracy": 0.8598, "macro_f1": 0.8594, "order": 2},
    {"model": "TF-IDF + Linear SVM",           "accuracy": 0.8641, "macro_f1": 0.8636, "order": 3},
    {"model": "TF-IDF + SVM, est. true labels", "accuracy": 0.891, "macro_f1": None,   "order": 4},
])

audit = pd.read_csv(f"{BASE_DIR}/audit_labels.csv").fillna("")
audit = audit[audit["my_label"] != ""].copy()
audit["verdict"] = np.select(
    [audit["my_label"] == "MIXED", audit["my_label"] == audit["label"]],
    ["Mixed / unclear", "Dataset label correct"], default="Dataset label WRONG")
audit["group"] = audit["group"].map({"both wrong": "Both models wrong", "both right": "Both models right"})
pd.crosstab(audit["group"], audit["verdict"])

verdict,Dataset label WRONG,Dataset label correct,Mixed / unclear
group,,,
Both models right,3,46,1
Both models wrong,71,22,7


## 6. Save
`utf-8-sig` encoding makes sure Persian text opens correctly in Power BI and Excel.

In [8]:
cols = ["review_id", "comment", "label", "sentiment", "sentiment_name", "split", "n_words",
        "length_bucket", "length_bucket_order", "mentions_late", "mentions_cold", "is_english",
        "cluster", "complaint_type", "predicted_name", "model_correct"]

reviews[cols].to_csv(f"{OUT_DIR}/reviews.csv", index=False, encoding="utf-8-sig")
food_mentions[["review_id", "food", "sentiment_name", "sentiment"]].to_csv(f"{OUT_DIR}/food_mentions.csv", index=False, encoding="utf-8-sig")
model_results.to_csv(f"{OUT_DIR}/model_results.csv", index=False, encoding="utf-8-sig")
audit[["comment", "label", "my_label", "group", "verdict"]].to_csv(f"{OUT_DIR}/audit.csv", index=False, encoding="utf-8-sig")

for f in sorted(os.listdir(OUT_DIR)):
    print(f"{f:20} {os.path.getsize(f'{OUT_DIR}/{f}') / 1e6:.1f} MB")

audit.csv            0.0 MB
food_mentions.csv    0.9 MB
model_results.csv    0.0 MB
reviews.csv          15.4 MB
